# 01 — Load Signals

Pulls `signals` table from micro-arb's Railway Postgres.

Schema: `implied_prob_market` = YES price, `implied_prob_real` = 1 - NO price (fair YES implied by NO token).

**V1**: `spread == implied_prob_market - implied_prob_real` (= YES + NO - 1).  
**C1**: no arb detection logic here. **V9**: config from file.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from src import config, db

cfg = config.load()
print('Config:', cfg)

In [ ]:
rows = db.query("""
    SELECT
        id,
        correlation_id,
        contract_id,
        direction,
        implied_prob_market   AS yes_price,
        implied_prob_real     AS implied_no_fair,
        spread,
        expected_edge,
        decision,
        rejection_reason,
        decision_timestamp_ns
    FROM signals
    ORDER BY decision_timestamp_ns ASC
""")

signals = pd.DataFrame(rows)
for col in ['yes_price', 'implied_no_fair', 'spread', 'expected_edge']:
    signals[col] = pd.to_numeric(signals[col])

print(f'Rows: {len(signals)}')
signals.head()

In [ ]:
# V1: spread = implied_prob_market - implied_prob_real
# Derivation: yes + no - 1 = yes_price + (1 - implied_no_fair) - 1 = yes_price - implied_no_fair
signals['spread_check'] = signals['yes_price'] - signals['implied_no_fair']
drift = (signals['spread'] - signals['spread_check']).abs()
bad = drift > 1e-4  # NUMERIC(8,6) precision
print(f'V1 violations (|drift| > 1e-4): {bad.sum()} / {len(signals)}')
if bad.sum() == 0:
    print('V1 OK')
else:
    print('V1 WARN — sample violations:')
    print(signals[bad][['correlation_id','yes_price','implied_no_fair','spread','spread_check']].head())

In [ ]:
# Decision breakdown
print(signals.groupby(['decision', 'rejection_reason']).size().rename('count').to_string())

In [ ]:
# Spread distribution for emitted signals
import matplotlib.pyplot as plt

emitted = signals[signals['decision'] == 'emitted']
print(f'Emitted: {len(emitted)}')
print(f'Spread  — mean: {emitted["spread"].abs().mean():.4f}  max: {emitted["spread"].abs().max():.4f}')
print(f'Edge    — mean: {emitted["expected_edge"].mean():.4f}  max: {emitted["expected_edge"].max():.4f}')

emitted['spread'].abs().hist(bins=50)
plt.title('|Spread| distribution — emitted signals')
plt.xlabel('|spread|')
plt.ylabel('count')
plt.tight_layout()
plt.savefig('../data/raw/spread_dist.png', dpi=120)
plt.show()

In [ ]:
signals.to_parquet('../data/raw/signals.parquet', index=False)
print('Saved data/raw/signals.parquet')